# Albert Einstein's Contributions to Physics
## A Detailed Lecture Series with Computational Demonstrations

This notebook walks through Einstein's revolutionary ideas, from special relativity to general relativity, the photoelectric effect, Brownian motion, and beyond. Where possible, we use symbolic computation (SymPy) to re-enact the mathematical verification of his theories.

---
## 1. Special Relativity (1905)

In his 1905 paper *"On the Electrodynamics of Moving Bodies"*, Einstein introduced:
- The principle of relativity: the laws of physics are the same in all inertial frames.
- The constancy of the speed of light $c$ for all observers.

**Key consequences:**
- Lorentz transformations replace Galilean transformations.
- Time dilation: $\Delta t' = \gamma \Delta t$.
- Length contraction: $L' = L/\gamma$.
- Relativity of simultaneity.
- Spacetime interval invariance: $ds^2 = -c^2 dt^2 + dx^2 + dy^2 + dz^2$.

**Lorentz transformation (boost along $x$):**
$$
t' = \gamma (t - \frac{v}{c^2}x), \quad x' = \gamma (x - vt), \quad y'=y, \quad z'=z
$$
where $\gamma = 1/\sqrt{1 - v^2/c^2}$.

In [ ]:
import sympy as sp

# Symbols
c, v, t, x, y, z = sp.symbols('c v t x y z', positive=True)
# Boost velocity v < c
gamma = 1/sp.sqrt(1 - v**2/c**2)

# Lorentz transformation (boost in x-direction)
t_prime = gamma*(t - v*x/c**2)
x_prime = gamma*(x - v*t)
y_prime = y
z_prime = z

# Verify invariance of spacetime interval
s2 = -c**2*t**2 + x**2 + y**2 + z**2
s2_prime = -c**2*t_prime**2 + x_prime**2 + y_prime**2 + z_prime**2
sp.simplify(s2_prime - s2)  # Should be 0

---
## 2. Photoelectric Effect (1905)

Einstein extended Planck's quantum hypothesis to light itself, proposing that light consists of quanta (photons) with energy $E = hf$. The photoelectric equation:
$$
K_{\max} = hf - \phi
$$
where $K_{\max}$ is the maximum kinetic energy of emitted electrons, $h$ is Planck's constant, $f$ the frequency of incident light, and $\phi$ the work function of the material. This explained the threshold frequency and the instantaneous emission, contradicting classical wave theory. Nobel Prize in Physics 1921.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example parameters
h = 4.135667662e-15   # eV·s
phi = 2.0             # eV (work function for sodium)
f = np.linspace(4.0e14, 8.0e14, 100)
Kmax = h*f - phi
Kmax[Kmax < 0] = 0

plt.figure(figsize=(8,5))
plt.plot(f, Kmax)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Maximum Kinetic Energy (eV)')
plt.title('Photoelectric Effect: K_max = hf - φ')
plt.grid(True)
plt.show()

---
## 3. Brownian Motion (1905)

Einstein's analysis of the erratic motion of pollen grains suspended in water provided convincing evidence for the atomic theory of matter. He derived the diffusion equation:
$$
\langle x^2 \rangle = 2Dt
$$
where $D$ is the diffusion coefficient. This linked the macroscopic observable displacement to Avogadro's number, later experimentally verified by Jean Perrin.

In [ ]:
# Simple 1D random walk illustration
np.random.seed(42)
steps = 1000
position = np.cumsum(np.random.choice([-1, 1], size=steps))
plt.figure(figsize=(8,4))
plt.plot(position)
plt.xlabel('Step number')
plt.ylabel('Displacement')
plt.title('1D Random Walk (Brownian Motion)')
plt.grid(True)
plt.show()

---
## 4. Mass–Energy Equivalence (1905)

In a brief follow‑up paper, Einstein derived the most famous equation in physics:
$$
E = mc^2
$$
This expresses the equivalence of mass and energy, and is the basis of nuclear energy and our understanding of stellar processes.

In [ ]:
# Symbolic demonstration: rest energy
m, c = sp.symbols('m c', positive=True)
E = m*c**2
E

---
## 5. General Relativity (1915)

Einstein's masterpiece, completed after a decade of work, describes gravity not as a force but as the curvature of spacetime caused by mass and energy. The Einstein field equations:
$$
G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2}g_{\mu\nu}R = \frac{8\pi G}{c^4}T_{\mu\nu}
$$

**Key predictions:**
- Precession of Mercury's perihelion (solved an existing anomaly).
- Deflection of starlight by the Sun (confirmed by Eddington in 1919).
- Gravitational redshift and black holes.
- Gravitational waves (directly detected by LIGO in 2015).

Karl Schwarzschild found the first exact solution in 1916, just months after Einstein's publication. The Schwarzschild metric describes the spacetime outside a spherically symmetric, non‑rotating mass. Below we use symbolic computation to verify that this metric indeed satisfies the vacuum field equations ($T_{\mu\nu}=0 \Rightarrow G_{\mu\nu}=0$).

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, sin, cos, simplify

# --- 1. Define basic symbols ---
t, r, theta, phi = symbols('t r theta phi')
M = symbols('M')

# --- 2. Define metric tensor g_{mu,nu} ---
g00 = -(1 - 2*M/r)
g11 = 1/(1 - 2*M/r)
g22 = r**2
g33 = r**2 * sin(theta)**2

g = Matrix([
    [g00, 0, 0, 0],
    [0, g11, 0, 0],
    [0, 0, g22, 0],
    [0, 0, 0, g33]
])
coords = [t, r, theta, phi]

# --- 3. Inverse metric ---
g_inv = g.inv()

# --- 4. Christoffel symbols ---
Gamma = sp.MutableDenseNDimArray.zeros(4, 4, 4)
for rho in range(4):
    for mu in range(4):
        for nu in range(4):
            sum_expr = 0
            for lam in range(4):
                term = (sp.diff(g[lam, mu], coords[nu]) +
                        sp.diff(g[lam, nu], coords[mu]) -
                        sp.diff(g[mu, nu], coords[lam]))
                sum_expr += g_inv[rho, lam] * term
            Gamma[rho, mu, nu] = simplify(sum_expr / 2)

# --- 5. Riemann tensor ---
Riemann = sp.MutableDenseNDimArray.zeros(4, 4, 4, 4)
for rho in range(4):
    for sigma in range(4):
        for mu in range(4):
            for nu in range(4):
                term1 = sp.diff(Gamma[rho, sigma, nu], coords[mu])
                term2 = sp.diff(Gamma[rho, sigma, mu], coords[nu])
                sum_terms = 0
                for lam in range(4):
                    sum_terms += (Gamma[rho, mu, lam] * Gamma[lam, sigma, nu] -
                                  Gamma[rho, nu, lam] * Gamma[lam, sigma, mu])
                Riemann[rho, sigma, mu, nu] = simplify(term1 - term2 + sum_terms)

# --- 6. Ricci tensor ---
Ricci = sp.MutableDenseNDimArray.zeros(4, 4)
for mu in range(4):
    for nu in range(4):
        Ricci[mu, nu] = simplify(sum(Riemann[rho, mu, rho, nu] for rho in range(4)))

# --- 7. Ricci scalar ---
Ricci_scalar = sum(g_inv[mu, nu] * Ricci[mu, nu] for mu in range(4) for nu in range(4))
Ricci_scalar = simplify(Ricci_scalar)

# --- 8. Einstein tensor ---
Einstein = sp.MutableDenseNDimArray.zeros(4, 4)
for mu in range(4):
    for nu in range(4):
        Einstein[mu, nu] = simplify(Ricci[mu, nu] - sp.Rational(1,2)*g[mu, nu]*Ricci_scalar)

# Display all components (should be zero)
for mu in range(4):
    for nu in range(4):
        if Einstein[mu, nu] != 0:
            print(f"G_{mu}{nu} = {Einstein[mu, nu]}")
        else:
            print(f"G_{mu}{nu} = 0")

The vanishing of all components confirms that the Schwarzschild solution is a vacuum solution of Einstein's equations.

**Visualizing curvature:** The non‑zero Riemann components represent tidal forces. For example, the radial component $R^t_{rtr} \propto 1/r^3$, showing how curvature fades with distance.

In [ ]:
# Print a few non-zero Riemann components (lowered indices)
Riemann_lowered = sp.MutableDenseNDimArray.zeros(4, 4, 4, 4)
for rho in range(4):
    for sigma in range(4):
        for mu in range(4):
            for nu in range(4):
                Riemann_lowered[rho, sigma, mu, nu] = simplify(
                    sum(g[rho, lam] * Riemann[lam, sigma, mu, nu] for lam in range(4)))

print("Non-zero Riemann components (lowered indices):")
for rho in range(4):
    for sigma in range(rho+1, 4):
        for mu in range(4):
            for nu in range(mu+1, 4):
                val = Riemann_lowered[rho, sigma, mu, nu]
                if val != 0:
                    print(f"R_{{{rho}{sigma}{mu}{nu}}} = {val}")

---
## 6. Cosmological Constant (1917)

To allow a static universe (before Hubble's discovery of expansion), Einstein added a term $\Lambda$ to his field equations:
$$
G_{\mu\nu} + \Lambda g_{\mu\nu} = \frac{8\pi G}{c^4} T_{\mu\nu}
$$
After the expansion was observed, he called $\Lambda$ his "biggest blunder". Modern cosmology, however, resurrects $\Lambda$ as the simplest explanation for dark energy.

---
## 7. Quantum Entanglement & EPR Paradox (1935)

With Podolsky and Rosen, Einstein published a paper arguing that quantum mechanics was incomplete. The EPR paradox highlighted the strange non‑locality of entangled states, which Einstein called "spooky action at a distance". This sparked decades of foundational debate, eventually leading to Bell's theorem and experimental confirmation of entanglement.

---
## 8. Bose–Einstein Statistics & Condensates (1924‑1925)

Einstein extended Bose's work on photon statistics to massive particles, predicting a new state of matter – the Bose–Einstein condensate – at ultra‑low temperatures. This was experimentally realized in 1995.

---
## 9. Unified Field Theory (later years)

Einstein spent the last 30 years of his life searching for a classical unified field theory that would combine gravity and electromagnetism. Though unsuccessful, his quest anticipated modern attempts at unification, such as string theory.

---
## Conclusion

Albert Einstein's contributions reshaped fundamental physics. From the quantum nature of light to the curvature of spacetime, his ideas continue to be tested and verified with ever‑increasing precision. This notebook only scratches the surface; each topic can be expanded into an entire lecture course.